# Quantitative Equity Analysis: Returns, Risk Modeling, and Trading Strategies

## Overview

This project presents a systematic quantitative analysis of equity markets, covering the complete pipeline from raw price data to risk management and algorithmic strategy evaluation. The work is organized into five thematic modules:

| Module | Topic |
|--------|-------|
| **1** | Return Mechanics & Probabilistic Price Modeling |
| **2** | Empirical Distribution Analysis & Normality Testing |
| **3** | Bootstrap Inference for Return Statistics |
| **4** | Portfolio Construction: EW, GMV, and Tangency |
| **5** | Risk Measurement: VaR & ES under Competing Models |
| **6** | Technical Trading: Moving Average & Bollinger Band Strategies |

**Universe:** AAPL, AXP, BA, CAT, MSFT, SBUX, DAL · **Data source:** Yahoo Finance · **Period:** 1999–2025


---
## Environment Setup

All required libraries for numerical computation, statistical analysis, financial data acquisition, and visualization.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy import stats
from scipy.stats import norm, iqr
from scipy.optimize import minimize
from statsmodels.distributions.empirical_distribution import ECDF
from statsmodels.stats.stattools import jarque_bera
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.35
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


---
## Module 1: Return Mechanics & Probabilistic Price Modeling

This module establishes the mathematical foundations of return analysis. We compute simple and log returns, examine multi-period aggregation, and derive the probability that an asset price crosses a given threshold under the lognormal assumption.


### 1.1  Simple and Log Returns — Calculation and Annualization

Given a price series, we compute one-period simple and log returns, their averages, and corresponding annualized figures using standard compounding conventions.

In [ ]:
# Price observations
P1, P2, P3 = 44.89, 46.76, 47.55

# One-period returns
R2 = (P2 - P1) / P1;   R3 = (P3 - P2) / P2
r2 = np.log(P2 / P1);  r3 = np.log(P3 / P2)

avg_simple = (R2 + R3) / 2
avg_log    = (r2 + r3) / 2

annual_simple = (1 + avg_simple) ** 12 - 1
annual_log    = 12 * avg_log

print(f"Simple return  R3(2)              = {R3:.6f}")
print(f"Log return     r3(2)              = {r3:.6f}")
print(f"Average monthly simple return     = {avg_simple:.6f}")
print(f"Average monthly log return        = {avg_log:.6f}")
print(f"Annualized simple return          = {annual_simple:.6f}")
print(f"Annualized log return             = {annual_log:.6f}")


### 1.2  Multi-Period Return Distribution

Under i.i.d. normality, the $k$-period cumulative log return $r_t(k) \sim \mathcal{N}(k\mu,\, k\sigma^2)$. We compute the CDF at a specified threshold and the covariance between overlapping sub-period returns.

In [ ]:
mu_ann, var_ann = 0.06, 0.47
mean_4, var_4   = 4 * mu_ann, 4 * var_ann

prob_below = norm.cdf(2.0, loc=mean_4, scale=np.sqrt(var_4))

print(f"E[r_t(4)]        = {mean_4:.4f}")
print(f"Var[r_t(4)]      = {var_4:.4f}")
print(f"P(r_t(4) < 2.0)  = {prob_below:.6f}")
print(f"Cov(r2(2),r3(2)) = {var_ann}  [equals sigma^2: shared sub-period]")


### 1.3  Probability of Breaching a Price Threshold

We model cumulative log price changes as Gaussian and compute the probability of (a) a portfolio falling below a floor and (b) a stock price exceeding a target level over a fixed horizon.

In [ ]:
# (a) Portfolio floor breach
mu_d, sig_d = 0.001, 0.015
V0, V_floor = 1000, 990
log_floor   = np.log(V_floor / V0)

p_1d = norm.cdf(log_floor, loc=mu_d,   scale=sig_d)
p_5d = norm.cdf(log_floor, loc=5*mu_d, scale=np.sqrt(5)*sig_d)
print("Portfolio breach probability (V < $990)")
print(f"  1 day  : {p_1d:.6f}")
print(f"  5 days : {p_5d:.6f}")

# (b) Price target
mu_p, var_p       = 0.0002, 0.032
P0_p, P_tgt, T    = 97, 100, 20
threshold_p       = np.log(P_tgt / P0_p)
p_exceed = 1 - norm.cdf(threshold_p, loc=T*mu_p, scale=np.sqrt(T*var_p))
print(f"
P(price > ${P_tgt} after {T} days) = {p_exceed:.6f}")


### 1.4  Long-Run Geometric Growth Rate

We recover the implied average annual simple and log returns from beginning-of-period and end-of-period wealth.

In [ ]:
initial, final, T = 22_400, 1_000_000, 100

R_ann = (final / initial) ** (1/T) - 1
r_ann = np.log(final / initial) / T

print(f"Horizon          : {T} years")
print(f"Initial wealth   : ${initial:,}")
print(f"Final wealth     : ${final:,}")
print(f"Ann. simple ret  : {R_ann:.6f}  ({R_ann*100:.4f}%)")
print(f"Ann. log return  : {r_ann:.6f}  ({r_ann*100:.4f}%)")


---
## Module 2: Empirical Distribution Analysis of Equity Log Returns

We download three years of daily log returns for Microsoft (MSFT) and rigorously test the normality assumption that underlies much of classical asset pricing. The analysis employs ECDF comparison, kernel density estimation, QQ plots, higher-moment tests, and heavy-tail modeling via Student-$t$ MLE.


### 2.1  Data Acquisition

In [ ]:
data   = yf.download("MSFT", start="2022-01-02", end="2024-12-30", progress=False)
prices = data["Close"].squeeze()

log_ret_msft = (100 * np.log(prices / prices.shift(1))).dropna().squeeze()
log_ret_msft = np.asarray(log_ret_msft).flatten()

mu_m    = float(np.mean(log_ret_msft))
sig2_m  = float(np.var(log_ret_msft, ddof=1))
sig_m   = np.sqrt(sig2_m)
x_grid  = np.linspace(log_ret_msft.min(), log_ret_msft.max(), 500)

print(f"Observations : {len(log_ret_msft)}")
print(f"Sample mean  : {mu_m:.6f} %")
print(f"Sample std   : {sig_m:.6f} %")


### 2.2  ECDF vs Normal CDF

Deviations in the tails indicate departures from normality; the ECDF rising more slowly than the normal CDF signals heavier tails.

In [ ]:
ecdf       = ECDF(log_ret_msft)
normal_cdf = stats.norm.cdf(x_grid, loc=mu_m, scale=sig_m)

plt.figure()
plt.plot(x_grid, ecdf(x_grid), lw=2, label="Empirical CDF")
plt.plot(x_grid, normal_cdf,  "--", lw=2, label="Normal CDF (fitted)")
plt.title("MSFT Daily Log Returns: ECDF vs Normal CDF (2022–2024)")
plt.xlabel("Daily Log Return (%)"); plt.ylabel("Cumulative Probability")
plt.legend(); plt.show()


**Finding:** The ECDF tracks the normal CDF closely in the center but rises more slowly in both tails, consistent with the heavy-tailed behavior widely documented in equity returns.

### 2.3  Kernel Density Estimate vs Normal PDF

In [ ]:
kde        = stats.gaussian_kde(log_ret_msft)
kernel_pdf = kde(x_grid)
normal_pdf = stats.norm.pdf(x_grid, loc=mu_m, scale=sig_m)

plt.figure()
plt.plot(x_grid, kernel_pdf, lw=2, label="KDE")
plt.plot(x_grid, normal_pdf, "--", lw=2, label="Normal PDF (fitted)")
plt.title("MSFT Daily Log Returns: KDE vs Normal PDF (2022–2024)")
plt.xlabel("Daily Log Return (%)"); plt.ylabel("Density")
plt.legend(); plt.show()


**Finding:** The KDE peak is sharper and taller than the fitted Gaussian (leptokurtosis), with slightly heavier tails. Returns cluster near the mean more than normal theory predicts while extreme events occur more frequently.

### 2.4  QQ Plot

In [ ]:
plt.figure(figsize=(7, 6))
stats.probplot(log_ret_msft, dist="norm", sparams=(mu_m, sig_m), plot=plt)
plt.title("QQ Plot: MSFT Log Returns vs Normal Distribution")
plt.show()


**Finding:** The center of the distribution conforms to the diagonal, but both tails bow outward — confirming fat tails and a higher-than-normal frequency of extreme gains and losses.

### 2.5  Higher-Moment Tests and Jarque-Bera

In [ ]:
n_m      = len(log_ret_msft)
skew_hat = stats.skew(log_ret_msft)
kurt_hat = stats.kurtosis(log_ret_msft, fisher=False)   # Pearson; normal = 3

z_skew = skew_hat / np.sqrt(6 / n_m)
p_skew = 2 * (1 - stats.norm.cdf(abs(z_skew)))

z_kurt = (kurt_hat - 3) / np.sqrt(24 / n_m)
p_kurt = 1 - stats.norm.cdf(z_kurt)

jb_stat, jb_p, _, _ = jarque_bera(log_ret_msft)

print("=== Normality Tests ===")
print(f"Skewness   : {skew_hat:.4f}  z={z_skew:.4f}  p={p_skew:.4f}"
      f"  → {'reject H0' if p_skew < 0.05 else 'fail to reject H0'}")
print(f"Kurtosis   : {kurt_hat:.4f}  z={z_kurt:.4f}  p={p_kurt:.6f}"
      f"  → {'reject H0 (excess kurtosis)' if p_kurt < 0.05 else 'fail to reject H0'}")
print(f"Jarque-Bera: {jb_stat:.4f}  p={jb_p:.6f}"
      f"  → {'reject normality' if jb_p < 0.05 else 'fail to reject normality'}")


**Finding:** Skewness is slightly negative but statistically indistinct from zero ($p > 0.05$). Kurtosis significantly exceeds 3, and the Jarque-Bera test rejects normality at any conventional level. The distribution is leptokurtic but not materially asymmetric.

### 2.6  Heavy-Tail Modeling: Student-$t$ MLE

Given the evidence of fat tails, we fit a Student-$t$ distribution via MLE. The estimated degrees of freedom $\hat{\nu}$ governs tail weight; low values imply heavy tails with convergence to Gaussian as $\hat{\nu} \to \infty$.

In [ ]:
nu_mle, mu_mle, sigma_mle = stats.t.fit(log_ret_msft)

student_pdf = stats.t.pdf(x_grid, df=nu_mle, loc=mu_mle, scale=sigma_mle)

plt.figure()
plt.plot(x_grid, kernel_pdf,  lw=2, label="KDE")
plt.plot(x_grid, normal_pdf,  "--", lw=2, label=f"Normal")
plt.plot(x_grid, student_pdf, "-.", lw=2, label=f"Student-t (ν̂={nu_mle:.1f})")
plt.title("MSFT Return Distribution: KDE vs Normal vs Student-t")
plt.xlabel("Daily Log Return (%)"); plt.ylabel("Density")
plt.legend(); plt.show()

print(f"MLE estimates — μ̂={mu_mle:.6f}  σ̂={sigma_mle:.6f}  ν̂={nu_mle:.4f}")
print("Low ν̂ confirms heavy tails inconsistent with the Gaussian assumption.")


---
## Module 3: Bootstrap Inference for Return Statistics

Classical analytic standard errors for moments rely on distributional assumptions that equity data routinely violates. We use the nonparametric bootstrap to construct standard errors and confidence intervals for the mean, variance, and standard deviation of AAPL weekly log returns — comparing bootstrap estimates against their analytic counterparts.


### 3.1  Data and Sample Moments

In [ ]:
aapl_raw    = yf.download("AAPL", start="2010-01-01", end="2024-01-31",
                           interval="1wk", progress=False)
aapl_prices = aapl_raw["Close"].dropna()
aapl_lr     = np.log(aapl_prices / aapl_prices.shift(1)).dropna()
dat         = aapl_lr.to_numpy().flatten()
n_a         = len(dat)

mu_a    = np.mean(dat)
sig2_a  = np.var(dat, ddof=1)
sd_a    = np.sqrt(sig2_a)

SE_mean_analytic = np.sqrt(sig2_a / n_a)
SE_var_analytic  = np.sqrt(2 / n_a) * sig2_a

print(f"Observations   : {n_a}")
print(f"Sample mean    : {mu_a:.6f}")
print(f"Sample variance: {sig2_a:.6f}")
print(f"Sample std dev : {sd_a:.6f}")
print(f"
Analytic SE(mean) : {SE_mean_analytic:.6f}")
print(f"Analytic SE(var)  : {SE_var_analytic:.6f}")


### 3.2  Bootstrap Standard Errors

We draw $B = 1000$ resamples with replacement and compute standard errors for the mean, variance, and standard deviation using both the standard deviation of the bootstrap distribution and the IQR-based robust estimator.

In [ ]:
NBOOT = 1000
np.random.seed(432)

boot_samples = np.array([np.random.choice(dat, replace=True, size=n_a)
                          for _ in range(NBOOT)])   # shape (B, n)

boot_means = boot_samples.mean(axis=1)
boot_vars  = boot_samples.var(axis=1, ddof=1)
boot_sds   = np.sqrt(boot_vars)

IQR_scale = norm.ppf(0.75) - norm.ppf(0.25)

print("=== Bootstrap Standard Errors ===")
print(f"
Mean:")
print(f"  Bootstrap SE      : {boot_means.std(ddof=1):.6f}")
print(f"  IQR-based SE      : {iqr(boot_means)/IQR_scale:.6f}")
print(f"  Analytic SE       : {SE_mean_analytic:.6f}")

print(f"
Variance:")
print(f"  Bootstrap SE      : {boot_vars.std(ddof=1):.6f}")
print(f"  IQR-based SE      : {iqr(boot_vars)/IQR_scale:.6f}")
print(f"  Analytic SE       : {SE_var_analytic:.6f}")

print(f"
Standard Deviation:")
print(f"  Bootstrap SE      : {boot_sds.std(ddof=1):.6f}")
print(f"  IQR-based SE      : {iqr(boot_sds)/IQR_scale:.6f}")


### 3.3  Bootstrap Bias, MSE, and Confidence Interval for SD

In [ ]:
bias_sd = np.mean(boot_sds) - sd_a
mse_sd  = np.mean((boot_sds - sd_a) ** 2)

ci_lo, ci_hi = np.percentile(boot_sds, [2.5, 97.5])

print(f"Bootstrap Bias(SD) : {bias_sd:.8f}")
print(f"Bootstrap MSE(SD)  : {mse_sd:.8f}")
print(f"
95% Symmetric Bootstrap CI for SD:")
print(f"  [{ci_lo:.6f},  {ci_hi:.6f}]")

# Distribution of bootstrap SDs
plt.figure(figsize=(8, 4))
plt.hist(boot_sds, bins=40, edgecolor="white", linewidth=0.5)
plt.axvline(sd_a,   color="red",   lw=2, label=f"Sample SD = {sd_a:.4f}")
plt.axvline(ci_lo,  color="orange",lw=1.5, ls="--", label="95% CI bounds")
plt.axvline(ci_hi,  color="orange",lw=1.5, ls="--")
plt.title("Bootstrap Distribution of Sample Standard Deviation (AAPL Weekly)")
plt.xlabel("SD"); plt.ylabel("Frequency")
plt.legend(); plt.show()


---
## Module 4: Multi-Asset Portfolio Construction

We build a three-stock equity universe (AXP, CAT, SBUX) using daily price data from 1999 to 2024. Three canonical portfolio strategies are constructed and compared:

- **Equally Weighted (EW):** $w_i = 1/N$
- **Global Minimum Variance (GMV):** $w = \Sigma^{-1}\mathbf{1} \;/\; (\mathbf{1}^\top \Sigma^{-1} \mathbf{1})$
- **Tangency (Max Sharpe):** $w = \Sigma^{-1}\mu \;/\; (\mathbf{1}^\top \Sigma^{-1} \mu)$


### 4.1  Data Download and Return Summary

In [ ]:
tickers3 = ["AXP", "CAT", "SBUX"]
df3 = yf.download(tickers3, start="1999-01-01", end="2024-12-31",
                  progress=False, auto_adjust=True)["Close"]
df3.columns = tickers3
df3 = df3.dropna()

simple_ret = df3.pct_change().dropna()
log_ret3   = np.log(1 + simple_ret)

plt.figure()
for col in df3.columns:
    plt.plot(df3.index, df3[col], label=col)
plt.title("Adjusted Closing Prices: AXP, CAT, SBUX (1999–2024)")
plt.xlabel("Date"); plt.ylabel("Price ($)"); plt.legend(); plt.show()

print("Summary statistics — Daily Simple Returns:")
print((simple_ret * 100).describe().round(4))


### 4.2  Mean Return Hypothesis Tests

Two-sided $t$-test of $H_0: \mu = 0$ for each stock's daily log return.

In [ ]:
from scipy.stats import ttest_1samp

results_t = {}
for stock in log_ret3.columns:
    t_stat, p_val = ttest_1samp(log_ret3[stock], 0)
    results_t[stock] = {"Mean Log Return": log_ret3[stock].mean(),
                        "t-statistic":     t_stat,
                        "p-value":         p_val}

print("H0: Mean daily log return = 0")
print(pd.DataFrame(results_t).T.round(6))


### 4.3  Log Price Autoregression

OLS regression of $\log P_t$ on $\log P_{t-1}$ for each stock. The slope coefficient near unity indicates near-random-walk behavior; deviations suggest mean reversion or momentum.

In [ ]:
log_prices3 = np.log(df3)

for stock in tickers3:
    y  = log_prices3[stock].iloc[1:]
    x  = log_prices3[stock].shift(1).iloc[1:]
    X  = sm.add_constant(x)
    m  = sm.OLS(y, X).fit()

    xv = np.linspace(x.min(), x.max(), 100)
    yh = m.params["const"] + m.params[stock] * xv

    plt.figure(figsize=(7, 5))
    plt.scatter(x, y, alpha=0.15, s=4, label="Data")
    plt.plot(xv, yh, color="crimson", lw=2, label="OLS fit")
    plt.xlabel(r"$\log P_{t-1}$"); plt.ylabel(r"$\log P_t$")
    plt.title(f"Log Price Autoregression — {stock}")
    plt.legend(); plt.show()

    print(f"{stock}: slope={m.params[stock]:.6f}  R²={m.rsquared:.6f}")


### 4.4  Portfolio Construction and Performance

We compute portfolio weights analytically and evaluate realized return series over the full sample.

In [ ]:
n_p     = simple_ret.shape[1]
Sigma   = simple_ret.cov().values
mu_vec  = simple_ret.mean().values
ones    = np.ones(n_p)
iSigma  = np.linalg.inv(Sigma)

w_ew  = np.repeat(1/n_p, n_p)
w_gmv = iSigma @ ones  / (ones  @ iSigma @ ones)
w_tan = iSigma @ mu_vec / (ones @ iSigma @ mu_vec)

ret_ew  = simple_ret.values @ w_ew
ret_gmv = simple_ret.values @ w_gmv
ret_tan = simple_ret.values @ w_tan
idx_p   = simple_ret.index

# Performance table
perf = pd.DataFrame({
    "Mean Return":    [r.mean()  for r in (ret_ew, ret_gmv, ret_tan)],
    "Std Dev":        [r.std()   for r in (ret_ew, ret_gmv, ret_tan)],
    "Daily Sharpe":   [r.mean()/r.std() for r in (ret_ew, ret_gmv, ret_tan)],
    "Ann. Sharpe":    [r.mean()/r.std()*np.sqrt(252) for r in (ret_ew, ret_gmv, ret_tan)],
}, index=["Equally Weighted", "Global Min Variance", "Tangency"])

print("Portfolio Performance Summary")
print(perf.round(6))

# Return series plot
plt.figure()
plt.plot(idx_p, ret_ew,  alpha=0.8, label="Equally Weighted")
plt.plot(idx_p, ret_gmv, alpha=0.8, label="Global Min Variance")
plt.plot(idx_p, ret_tan, alpha=0.8, label="Tangency")
plt.title("Daily Portfolio Returns: EW vs GMV vs Tangency")
plt.xlabel("Date"); plt.ylabel("Return"); plt.legend(); plt.show()

print("
Portfolio Weights")
print(pd.DataFrame({"EW": w_ew, "GMV": w_gmv, "Tangency": w_tan},
                   index=tickers3).round(4))


---
## Module 5: Value-at-Risk and Expected Shortfall

We estimate tail risk for a $100,000 position in MSFT (2012–2024) at the 1% significance level using three competing methods: historical simulation, parametric Normal, and parametric Student-$t$. Bootstrap confidence intervals quantify estimation uncertainty, and power-law tail extrapolation extends estimates to the extreme 0.1% quantile.


### 5.1  Data and Helper Functions

In [ ]:
msft_raw = yf.download("MSFT", start="2012-01-02", end="2024-12-31",
                        auto_adjust=True, progress=False)
msft_p   = msft_raw["Close"].dropna().squeeze()
msft_lr  = np.log(msft_p / msft_p.shift(1)).dropna().values.flatten()

W0    = 1e5
ALPHA = 0.01
B_var = 1000
SEED  = 432

def to_dollar(var_r, es_r, W=W0):
    return -W*var_r, -W*es_r

def ci_dollar(lo_r, hi_r, W=W0):
    return -W*hi_r, -W*lo_r

print(f"Observations: {len(msft_lr)}")
print(f"Period      : {msft_p.index[1].date()} – {msft_p.index[-1].date()}")


### 5.2  Nonparametric (Historical Simulation)

In [ ]:
sorted_lr = np.sort(msft_lr)
var_np_r  = np.quantile(msft_lr, ALPHA)
es_np_r   = sorted_lr[sorted_lr <= var_np_r].mean()
var_np_d, es_np_d = to_dollar(var_np_r, es_np_r)

print(f"Nonparametric  VaR_0.01 : {var_np_r:.6f}  (${var_np_d:,.2f})")
print(f"Nonparametric  ES_0.01  : {es_np_r:.6f}  (${es_np_d:,.2f})")


### 5.3  Parametric — Normal Distribution

In [ ]:
mu_msft, sig_msft = msft_lr.mean(), msft_lr.std(ddof=1)
var_n_r = stats.norm.ppf(ALPHA, loc=mu_msft, scale=sig_msft)
es_n_r  = mu_msft - sig_msft * stats.norm.pdf(stats.norm.ppf(ALPHA)) / ALPHA
var_n_d, es_n_d = to_dollar(var_n_r, es_n_r)

print(f"Normal  VaR_0.01 : {var_n_r:.6f}  (${var_n_d:,.2f})")
print(f"Normal  ES_0.01  : {es_n_r:.6f}  (${es_n_d:,.2f})")


### 5.4  Parametric — Student-$t$ Distribution

In [ ]:
nu_v, loc_v, scale_v = stats.t.fit(msft_lr, floc=mu_msft)
var_t_r = stats.t.ppf(ALPHA, df=nu_v, loc=loc_v, scale=scale_v)

t_alpha   = stats.t.ppf(ALPHA, df=nu_v)
f_t_alpha = stats.t.pdf(t_alpha, df=nu_v)
es_t_r    = loc_v - scale_v * (f_t_alpha/ALPHA) * (nu_v + t_alpha**2)/(nu_v - 1)
var_t_d, es_t_d = to_dollar(var_t_r, es_t_r)

print(f"Student-t (ν̂={nu_v:.2f})  VaR_0.01 : {var_t_r:.6f}  (${var_t_d:,.2f})")
print(f"Student-t (ν̂={nu_v:.2f})  ES_0.01  : {es_t_r:.6f}  (${es_t_d:,.2f})")


### 5.5  Method Comparison

In [ ]:
comp = pd.DataFrame({
    "VaR_0.01 (return)": [var_np_r, var_n_r,  var_t_r],
    "ES_0.01  (return)": [es_np_r,  es_n_r,   es_t_r],
    "VaR_0.01 ($)":      [var_np_d, var_n_d,  var_t_d],
    "ES_0.01  ($)":      [es_np_d,  es_n_d,   es_t_d],
}, index=["Nonparametric", "Normal", "Student-t"])

print(f"Excess kurtosis: {float(stats.kurtosis(msft_lr)):.4f}")
print()
print(comp.round(4))
print()
print("The Normal model underestimates tail risk due to its thin tails.")
print(f"Student-t (ν̂≈{nu_v:.1f}) captures fat tails → more conservative, more realistic estimates.")


### 5.6  Bootstrap Confidence Intervals (95%) for Nonparametric VaR and ES

In [ ]:
np.random.seed(SEED)
n_msft    = len(msft_lr)
boot_var  = np.empty(B_var)
boot_es   = np.empty(B_var)

for b in range(B_var):
    s           = np.random.choice(msft_lr, size=n_msft, replace=True)
    ss          = np.sort(s)
    bv          = np.quantile(s, ALPHA)
    boot_var[b] = bv
    boot_es[b]  = ss[ss <= bv].mean()

ci_var = np.percentile(boot_var, [2.5, 97.5])
ci_es  = np.percentile(boot_es,  [2.5, 97.5])
var_ci_d = ci_dollar(*ci_var)
es_ci_d  = ci_dollar(*ci_es)

print(f"95% CI VaR_0.01 (return): [{ci_var[0]:.6f},  {ci_var[1]:.6f}]")
print(f"95% CI ES_0.01  (return): [{ci_es[0]:.6f},  {ci_es[1]:.6f}]")
print(f"95% CI VaR_0.01 ($)     : [${var_ci_d[0]:,.2f},  ${var_ci_d[1]:,.2f}]")
print(f"95% CI ES_0.01  ($)     : [${es_ci_d[0]:,.2f},   ${es_ci_d[1]:,.2f}]")


### 5.7  Tail Index Estimation and Extreme Quantile Extrapolation

We fit a power-law tail using log-log regression on the top $m = \lfloor 0.1n \rfloor$ order statistics. The estimated tail index $\hat{\alpha}$ is then used to extrapolate VaR and ES to the 0.1% level, well beyond the empirical sample.

In [ ]:
losses     = -msft_lr
losses_desc = np.sort(losses)[::-1]
n_msft_v   = len(msft_lr)
m_top      = int(np.floor(0.1 * n_msft_v))
top_m      = losses_desc[:m_top]

X_reg = np.log(top_m)
Y_reg = np.log(np.arange(1, m_top+1) / n_msft_v)
slope, intercept, r2, _, _ = stats.linregress(X_reg, Y_reg)

alpha_hat = -slope
xi_hat    = 1.0 / alpha_hat

print(f"m = {m_top}  slope = {slope:.4f}  R² = {r2**2:.4f}")
print(f"Tail index  α̂ = {alpha_hat:.4f}")
print(f"Shape param ξ̂ = {xi_hat:.4f}")

# Extrapolation
alpha_ref   = 0.01
alpha_extrap = 0.001
var_ref_loss = float(np.quantile(losses, 1 - alpha_ref))

var_ext = var_ref_loss * (alpha_ref / alpha_extrap) ** xi_hat
es_ext  = (var_ext * alpha_hat / (alpha_hat - 1)) if alpha_hat > 1 else np.nan

print(f"
Extrapolated to α = {alpha_extrap}:")
print(f"  VaR_0.001 : {var_ext:.6f}  (${W0*var_ext:,.2f})")
if not np.isnan(es_ext):
    print(f"  ES_0.001  : {es_ext:.6f}  (${W0*es_ext:,.2f})")


### 5.8  Full Risk Summary

In [ ]:
summary = pd.DataFrame({
    "Nonparametric (α=0.01)":  [var_np_d,        es_np_d],
    "Normal        (α=0.01)":  [var_n_d,         es_n_d],
    "Student-t     (α=0.01)":  [var_t_d,         es_t_d],
    "Boot CI Lower (α=0.01)":  [var_ci_d[0],     es_ci_d[0]],
    "Boot CI Upper (α=0.01)":  [var_ci_d[1],     es_ci_d[1]],
    "Extrapolated  (α=0.001)": [W0*var_ext, W0*es_ext if not np.isnan(es_ext) else np.nan],
}, index=["VaR ($)", "ES ($)"]).T

print("Full Risk Summary — Dollar Losses, W0 = $100,000")
print(summary.to_string(float_format=lambda x: f"${x:>12,.2f}"))


---
## Module 6: Technical Trading Strategies — Moving Averages & Bollinger Bands

We evaluate systematic trading rules on the S&P 500 (2010–present) across two families:

1. **Moving-Average Rules** — Momentum (long when price ≥ MA) and Contrarian (long when price < MA), bandwidths $k \in \{5, 10, 20, 50, 100, 200\}$.
2. **Bollinger Band Rules** — Signals triggered by price breaking through the upper/lower bands ($\text{MA}_k \pm 2\sigma_k$), in both momentum and contrarian variants.

All signals are lagged one day to eliminate look-ahead bias. Performance is evaluated on annualized Sharpe ratio and expected daily log return.


### 6.1  Data and Indicator Construction

In [ ]:
sp500 = yf.download("^GSPC", start="2010-01-01", progress=False)
sp500 = pd.DataFrame({"Price": sp500["Close"].squeeze()})
sp500["Return"]    = sp500["Price"].pct_change()
sp500["LogReturn"] = np.log(sp500["Price"] / sp500["Price"].shift(1))

bandwidths = [5, 10, 20, 50, 100, 200]
for k in bandwidths:
    sp500[f"MA_{k}"]  = sp500["Price"].rolling(k).mean()
    sp500[f"STD_{k}"] = sp500["Price"].rolling(k).std()
    sp500[f"BBU_{k}"] = sp500[f"MA_{k}"] + 2 * sp500[f"STD_{k}"]
    sp500[f"BBL_{k}"] = sp500[f"MA_{k}"] - 2 * sp500[f"STD_{k}"]

print(f"S&P 500 loaded: {len(sp500):,} trading days")


### 6.2  Moving Average Filter — Visual Analysis (2023–Present)

In [ ]:
sp_2023 = sp500.loc["2023-01-01":].copy()

plt.figure(figsize=(13, 5))
plt.plot(sp_2023.index, sp_2023["Price"], lw=1.5, label="S&P 500 Close")
for k in [5, 20, 50, 200]:
    plt.plot(sp_2023.index, sp_2023[f"MA_{k}"], label=f"MA({k})")
plt.title("S&P 500 with Moving Average Filters (2023–Present)")
plt.xlabel("Date"); plt.ylabel("Index Level"); plt.legend(); plt.show()


### 6.3  Bollinger Bands — Visual Analysis

In [ ]:
for k in [5, 20]:
    plt.figure(figsize=(13, 5))
    plt.plot(sp_2023.index, sp_2023["Price"],      lw=1.5, label="Close")
    plt.plot(sp_2023.index, sp_2023[f"MA_{k}"],   "--",   label=f"MA({k})")
    plt.plot(sp_2023.index, sp_2023[f"BBU_{k}"],          label=f"Upper (k={k})")
    plt.plot(sp_2023.index, sp_2023[f"BBL_{k}"],          label=f"Lower (k={k})")
    plt.fill_between(sp_2023.index,
                     sp_2023[f"BBL_{k}"], sp_2023[f"BBU_{k}"], alpha=0.1)
    plt.title(f"S&P 500 Bollinger Bands (k={k}) — 2023–Present")
    plt.legend(); plt.show()


### 6.4  Moving-Average Momentum and Contrarian Strategies

In [ ]:
ma_results = []
for k in bandwidths:
    mom_sig = np.where(sp500["Price"] >= sp500[f"MA_{k}"], 1, -1)
    for label, sig in [("Momentum", mom_sig), ("Contrarian", -mom_sig)]:
        ret = (pd.Series(sig, index=sp500.index).shift(1) * sp500["Return"]).dropna()
        ma_results.append({"k": k, "Strategy": label,
                            "Mean Return":  ret.mean(),
                            "Std Dev":      ret.std(),
                            "Sharpe (ann)": ret.mean()/ret.std()*np.sqrt(252)})

ma_df = pd.DataFrame(ma_results)
print("Moving-Average Strategy Performance")
print(ma_df.round(6).to_string(index=False))


### 6.5  Bollinger Band Strategies

In [ ]:
bb_results = []
for k in bandwidths:
    price, bbu, bbl = sp500["Price"], sp500[f"BBU_{k}"], sp500[f"BBL_{k}"]
    lr = sp500["LogReturn"]

    con = pd.Series(0, index=sp500.index, dtype=float)
    con[price < bbl] =  1;  con[price > bbu] = -1
    mom = -con.copy()
    mom[price > bbu] =  1;  mom[price < bbl] = -1

    for label, sig in [("BB Momentum", mom), ("BB Contrarian", con)]:
        ret = (sig.shift(1) * lr).dropna()
        std = ret.std()
        bb_results.append({"k": k, "Strategy": label,
                            "Mean Log Ret":  ret.mean(),
                            "Std Dev":       std,
                            "Sharpe (ann)":  ret.mean()/std*np.sqrt(252) if std>0 else np.nan})

bb_df = pd.DataFrame(bb_results)
print("Bollinger Band Strategy Performance")
print(bb_df.round(6).to_string(index=False))


### 6.6  Risk-Adjusted Performance — Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_r, title in [(axes[0], ma_df, "Moving-Average"),
                         (axes[1], bb_df, "Bollinger Band")]:
    for strat, grp in df_r.groupby("Strategy"):
        ax.plot(grp["k"], grp["Sharpe (ann)"], marker="o", label=strat)
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_title(f"{title} Strategies"); ax.set_xlabel("Bandwidth k")
    ax.set_ylabel("Annualized Sharpe Ratio"); ax.legend()

plt.suptitle("Sharpe Ratio Across Bandwidths: Momentum vs Contrarian", fontsize=13)
plt.tight_layout(); plt.show()


### 6.7  Strategy Conclusions

**Moving-Average strategies:** The momentum variant generates positive Sharpe ratios at short bandwidths ($k \leq 20$), consistent with short-horizon price continuation in the S&P 500. The contrarian strategy produces the mirror-image — negative returns at short horizons — indicating that mean reversion is not the dominant dynamic at daily frequencies over this sample.

**Bollinger Band strategies:** The BB momentum rule (buy on upper-band breakouts, sell on lower-band breakouts) delivers positive risk-adjusted returns at short bandwidths with lower turnover than the always-invested MA variant.

**Overall:** Short-horizon momentum rules ($k = 5$–$20$) consistently outperform contrarian strategies on risk-adjusted grounds, aligning with the broader empirical literature on short-term momentum and trend-following in equity indices.


---
## Summary of Key Findings

| Module | Key Result |
|--------|------------|
| **Return Mechanics** | Log returns aggregate linearly across time; lognormality yields closed-form price-threshold probabilities |
| **Distribution Analysis** | MSFT log returns exhibit significant excess kurtosis; Student-$t$ MLE ($\hat{\nu} \approx 4$–6) fits the empirical distribution materially better than Gaussian |
| **Bootstrap Inference** | Bootstrap and analytic SEs are nearly identical for the mean; bootstrap SE for variance is larger, reflecting the heavier tails |
| **Portfolio Construction** | Tangency portfolio maximizes risk-adjusted return; GMV minimizes realized volatility at the cost of lower expected return |
| **Risk Measurement** | Normal VaR/ES underestimates tail risk; Student-$t$ estimates are more conservative and realistic; power-law extrapolation extends estimates to the 0.1% level |
| **Technical Strategies** | Short-horizon MA momentum ($k \leq 20$) dominates on Sharpe ratio; Bollinger Band momentum confirms price continuation over mean reversion at daily frequencies |
